In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, BatchNormalization, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import numpy as np
import re
import scipy.io


infile_path = 'Input10dB.csv'
df = pd.read_csv(infile_path)

# Extract the 'xI_real' and 'xI_imag' columns
y = df[['xI_real', 'xI_imag']]

output_mat_file_path = '/home/luke_mccubbin1/python_scripts/0210Dataset/IQ_input5dB.mat'


# Extract the 'xI_real' and 'xI_imag' columns
df['xI_complex'] = df['xI_real'] + 1j * df['xI_imag']

# Convert to NumPy array
data_to_save = {'MatlabInput': df['xI_complex'].values}

# Save to .mat file
scipy.io.savemat(output_mat_file_path, data_to_save)

print(f"Data successfully saved to {output_mat_file_path}")


# #Load the training data
outfile_path = 'Input10dB.csv'
df = pd.read_csv(outfile_path)

X = df[['yout_real', 'yout_imag']]


output_mat_file_path = '/home/luke_mccubbin1/python_scripts/0210Dataset/IQ_output5dB.mat'


# Extract the 'xI_real' and 'xI_imag' columns
df['yout_complex'] = df['yout_real'] + 1j * df['yout_imag']

# Convert to NumPy array
data_to_save = {'MatlabOuttrain5dB': df['yout_complex'].values}

# Save to .mat file
scipy.io.savemat(output_mat_file_path, data_to_save)

print(f"Data successfully saved to {output_mat_file_path}")


print("Shape of y:", y.shape)
print("Shape of X:", X.shape)

print('NN input files recieved')

2026-04-09 17:03:57.351655: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-09 17:03:57.365837: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775772237.380998 2344956 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775772237.385865 2344956 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775772237.398331 2344956 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

Data successfully saved to /home/luke_mccubbin1/python_scripts/0210Dataset/IQ_input5dB.mat


KeyError: "None of [Index(['yout_real', 'yout_imag'], dtype='object')] are in the [columns]"

In [ ]:
#NN model

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)



model = Sequential([
    Input(shape=(2,)),  # Explicitly defining the input shape
    Dense(32, activation='relu'),
    Dense(128, activation='relu'),
    Dense(64, activation='relu'),
    Dense(2)  # Output layer with 2 neurons for xI_real and xI_imag
])

# Compile the model
model.compile(optimizer='adam', loss='mean_squared_error', metrics=['accuracy'])

# Train the model
history = model.fit(X_train, y_train, epochs=5, validation_split=0.2, verbose=1)

# Evaluate the model
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=1)

y_pred = model.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)

print(f'Test Loss: {test_loss}')
print(f'Test accuracy: {test_accuracy}')
print(f'Test RMSE: {rmse}')

# Plot loss vs epoch
plt.figure(figsize=(8,5))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Loss vs Epoch')
plt.legend()
plt.grid(True)
plt.show()

print("Training complete")


# Function to parse complex numbers, including those with scientific notation from excel csv file
def parse_complex_number(s):
    s = s.replace("i", "j")  
    # Updated regex to handle scientific notation
    match = re.match(r"([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)([-+]\d*\.?\d+(?:[eE][-+]?\d+)?)j", s.replace(" ", ""))
    if match:
        real_part = float(match.group(1))
        imag_part = float(match.group(2))
        return real_part, imag_part
    else:
        raise ValueError(f"Malformed complex number: {s}")

# Load the new CSV file for predictions
# predict_file_path = '/home/luke_mccubbin1/python_scripts/0113Datatest/IQout_data_Pin10dBm_modTest.csv'
# df_predict = pd.read_csv(predict_file_path)

# # Use function to separate the real and imaginary parts of the yout complex numbers
# df_predict[['yout_real', 'yout_imag']] = df_predict['yout'].apply(
#     lambda x: pd.Series(parse_complex_number(x))
# )

# Load the training data
#outfile_path = 'IQout_data_Pin10dBm_mod.csv'
outfile_path = 'predist_5dBtest.csv'
df = pd.read_csv(outfile_path)

# Prepare input for prediction
X_new = df[['xI_real', 'xI_imag']]

# Predict using the model
y_new_pred = model.predict(X_new)  # Ensure this returns a 2D array

# Create the DataFrame with predictions
df_predict = pd.DataFrame({
    'xI_predict_real': y_new_pred[:, 0],
    'xI_predict_imag': y_new_pred[:, 1]
})

# Create complex numbers
df_predict['xI_predict'] = (df_predict['xI_predict_real'] + 1j * df_predict['xI_predict_imag']).astype(np.complex128)

# Prepare data for MATLAB file generation
data_to_save = {'xI_predict': df_predict['xI_predict'].values}


output_mat_file_path = '/home/luke_mccubbin1/python_scripts/0210Dataset/IQ5dB_predist.mat'
scipy.io.savemat(output_mat_file_path, data_to_save)

print(f'Predictions saved to {output_mat_file_path}')